# Fine-tune Qwen2.5-1.5B cho Hệ thống Vận hành Ngân hàng BankOps trên Google Colab T4 (Miễn phí)

Notebook này hướng dẫn tinh chỉnh mô hình **Qwen2.5-1.5B-Instruct** cho nghiệp vụ ngân hàng BankOps (quy trình Maker-Checker, đối soát sổ cái, xuất nhập kho phôi thẻ/sổ séc, bảo mật hash chain).

### Ưu điểm của mô hình và cấu hình này:
- **Mô hình**: `Qwen/Qwen2.5-1.5B-Instruct` - khả năng tiếng Việt và làm theo chỉ dẫn (instruction following) hàng đầu trong phân khúc nhỏ.
- **Tài nguyên cần thiết**: Hoạt động hoàn hảo trên GPU Google Colab T4 miễn phí (16GB VRAM, chỉ sử dụng ~3.5GB VRAM).
- **Thời gian huấn luyện**: ~10 đến 15 phút.
- **Triển khai sau huấn luyện**: Xuất ra định dạng GGUF `Q4_K_M` (~1.1 GB), chạy mượt mà trên máy **KHÔNG CÓ GPU** (chỉ dùng CPU, tiêu tốn < 1.5 GB RAM) thông qua Ollama hoặc llama.cpp.

In [ ]:
# Bước 1: Cài đặt Unsloth và các thư viện cần thiết
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers
print("✓ Cài đặt thư viện hoàn tất!")

In [ ]:
# Bước 2: Load Model Qwen2.5-1.5B 4-bit
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detect: Float16 for T4
load_in_4bit = True # Tiết kiệm VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✓ Đã tải mô hình Qwen2.5-1.5B thành công!")

In [ ]:
# Bước 3: Thêm LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✓ Đã cấu hình LoRA adapters!")

In [ ]:
# Bước 4: Chuẩn bị Dataset nghiệp vụ BankOps
from datasets import load_dataset

# Tải dataset trực tiếp hoặc từ train.jsonl
dataset = load_dataset("json", data_files={"train": "train.jsonl"}, split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Số mẫu huấn luyện:", len(dataset))
print("Mẫu đầu tiên:", dataset[0]["text"][:300])

In [ ]:
# Bước 5: Huấn luyện với SFTTrainer
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("✓ Huấn luyện hoàn tất!")

In [ ]:
# Bước 6: Kiểm thử suy luận (Inference)
FastLanguageModel.for_inference(model)

prompt = """<|im_start|>system
Bạn là trợ lý BankOps, trả lời bằng tiếng Việt, ngắn gọn và có bước thực hiện theo đúng chuẩn mực nghiệp vụ ngân hàng.<|im_end|>
<|im_start|>user
Giao dịch viên tạo phiếu thu tiền mặt thì khi nào số dư trong két sắt mới được tăng lên?<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# Bước 7: Xuất mô hình sang định dạng GGUF để chạy CPU không cần GPU
model.save_pretrained_gguf("bankops_qwen2.5_1.5b_gguf", tokenizer, quantization_method = "q4_k_m")
print("✓ Đã xuất file GGUF: bankops_qwen2.5_1.5b_gguf/qwen2.5-1.5b-q4_k_m.gguf")

# Tải file về máy cá nhân hoặc lưu vào Google Drive:
# from google.colab import files
# files.download("bankops_qwen2.5_1.5b_gguf/qwen2.5-1.5b-q4_k_m.gguf")